# Clean Full Enriched: 3 Algorithms (Simple Debug Subset)

Ноутбук собирает быстрый поддатасет из `clean_full_enriched` и гоняет 3 алгоритма
для отладки ограничений (включая новые объемные ограничения).


In [ ]:
from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path

import pandas as pd

from flowopt.backend import CleanEnrichedSimpleBuildConfig, build_clean_enriched_simple_dataset
from flowopt.pipeline import DATA_ROOT, run_real_gap_vrp, run_real_milp, run_real_genetic

REPO_ROOT = Path.cwd().resolve()
BASE_DATASET_PATH = DATA_ROOT / "real" / "clean_full_enriched" / "dataset_real_spb_clean_full_enriched.json"
SIMPLE_DIR = DATA_ROOT / "real" / "clean_full_enriched" / "simple"
DATASET_PATH = SIMPLE_DIR / "dataset_real_spb_clean_full_enriched_simple.json"
SUMMARY_PATH = SIMPLE_DIR / "summary_real_spb_clean_full_enriched_simple.json"

print("BASE_DATASET_PATH:", BASE_DATASET_PATH)
print("exists:", BASE_DATASET_PATH.exists())


In [ ]:
build_summary = build_clean_enriched_simple_dataset(
    base_dataset_path=BASE_DATASET_PATH,
    out_dataset_path=DATASET_PATH,
    out_summary_path=SUMMARY_PATH,
    config=CleanEnrichedSimpleBuildConfig(
        max_tasks=12,
        max_agents=64,
        max_task_pool=300,
    ),
)

print("DATASET_PATH:", DATASET_PATH)
print("exists:", DATASET_PATH.exists())
build_summary


In [ ]:
results = [
    run_real_gap_vrp(
        dataset_path=DATASET_PATH,
        step1_method="greedy",
        gap_iter=6,
        max_agents=32,
        use_repair=True,
        show_progress=False,
        verbose=False,
    ),
    run_real_milp(
        dataset_path=DATASET_PATH,
        time_limit_sec=20,
        unassigned_penalty=1e6,
        show_progress=False,
    ),
    run_real_genetic(
        dataset_path=DATASET_PATH,
        population_size=16,
        generations=24,
        generation_scale=0.5,
        elite_size=4,
        seed=42,
        show_progress=False,
    ),
]


In [ ]:
summary = pd.DataFrame([m.as_dict() for m in results])
cols = [
    "algorithm",
    "feasible",
    "all_checks_ok",
    "assigned_routes",
    "unassigned_tasks",
    "active_agents",
    "transport_work_ton_km",
    "runtime_sec",
    "payload_utilization_pct_avg",
    "volume_utilization_pct_avg",
    "object_mass_overload_count",
    "object_volume_overload_count",
    "deadhead_km",
    "deadhead_share_pct",
]
for c in cols:
    if c not in summary.columns:
        summary[c] = None
summary[cols]


In [ ]:
out_dir = REPO_ROOT / "notebooks" / "local" / "clean_full_enriched"
out_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_json = out_dir / f"simple_3algo_{stamp}.json"
out_csv = out_dir / f"simple_3algo_{stamp}.csv"

payload = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "dataset_path": str(DATASET_PATH),
    "build_summary": build_summary,
    "results": [m.as_dict() for m in results],
}
out_json.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
summary.to_csv(out_csv, index=False)
print("saved:", out_json)
print("saved:", out_csv)
